# 02 — Staging com DuckDB (exercício)

**Objetivo:** praticar SQL em DuckDB e extração de campos JSON.

Execute os notebooks em ordem.

## Onde estamos no pipeline

![Star schema alvo](../docs/images/image.png)

Este é o estado final que vamos atingir ao terminar a sessão: uma fato `fact_report` com chaves estrangeiras para quatro dimensões. Cada notebook contribui com uma camada do caminho até esse esquema.

Posição deste notebook (camada **silver / staging**):

> Fonte → Bronze → **Staging** → Dim/Fact → Checks → Profiling → Marts/Views → Dashboard

Aqui você lê o Parquet bronze no DuckDB, aplica casts de tipo e abre os campos `*_json` em colunas tabulares. As tabelas `stg_*` ainda não são as dimensões finais — elas são a base limpa que o notebook 03 vai usar para criar `fact_report` e as `dim_*`.

In [ ]:
# Parametros usados pelo Papermill/Airflow.
run_date = None
project_root = None

from pathlib import Path
import sys

for candidato in [Path.cwd(), *Path.cwd().parents]:
    caminhos_common = [
        candidato / "_common.py",
        candidato / "aula02" / "_common.py",
        candidato / "exercicios" / "_common.py",
        candidato / "sessao-02-data-architecture" / "_common.py",
        candidato / "sessao-02-data-architecture" / "exercicios" / "_common.py",
    ]
    encontrado = next((p for p in caminhos_common if p.exists()), None)
    if encontrado:
        sys.path.insert(0, str(encontrado.parent))
        break
else:
    raise FileNotFoundError("Nao encontrei _common.py a partir do diretorio atual.")

from _common import configurar_paths, conectar_duckdb
paths = configurar_paths(project_root)
globals().update(paths)

print(f"Raiz do projeto: {PROJECT_ROOT}")
print(f"Banco DuckDB: {DB_PATH}")


## Exercício
Complete as tabelas `stg_*` usando `json_extract_string`, casts e aliases.

In [ ]:
con = conectar_duckdb(DB_PATH)
print("Conexão DuckDB aberta.")

In [ ]:
bronze_path = str(BRONZE_PARQUET).replace("'", "''")
con.execute(f"CREATE OR REPLACE VIEW bronze AS SELECT * FROM read_parquet('{bronze_path}');")

con.sql("""
CREATE OR REPLACE TABLE stg_reports AS
SELECT
    CAST(id AS BIGINT) AS report_id,
    title,
    LOWER(NULLIF(substate, '')) AS substate,
    visibility,
    "has_bounty?" AS has_bounty,
    CAST(vote_count AS INTEGER) AS vote_count,
    CAST(created_at AS TIMESTAMP) AS created_at,
    CAST(disclosed_at AS TIMESTAMP) AS disclosed_at,
    reporter_json,
    weakness_json,
    team_json,
    structured_scope_json,
    vulnerability_information
FROM bronze;
""")

con.sql("""
CREATE OR REPLACE TABLE stg_reporter AS
SELECT DISTINCT
    reporter_json,
    json_extract_string(reporter_json, '$.username') AS username,
    CAST(json_extract_string(reporter_json, '$.verified') AS BOOLEAN) AS verified
FROM bronze;
""")

con.sql("""
CREATE OR REPLACE TABLE stg_team AS
SELECT DISTINCT
    team_json,
    CAST(json_extract_string(team_json, '$.id') AS BIGINT) AS id,
    json_extract_string(team_json, '$.handle') AS handle,
    CAST(json_extract_string(team_json, '$.offers_bounties') AS BOOLEAN) AS offers_bounties
FROM bronze;
""")

con.sql("""
CREATE OR REPLACE TABLE stg_weakness AS
SELECT DISTINCT
    weakness_json,
    CAST(json_extract_string(weakness_json, '$.id') AS BIGINT) AS id,
    json_extract_string(weakness_json, '$.name') AS name
FROM bronze;
""")

con.sql("""
CREATE OR REPLACE TABLE stg_asset AS
SELECT DISTINCT
    structured_scope_json,
    json_extract_string(structured_scope_json, '$.asset_identifier') AS asset_identifier,
    json_extract_string(structured_scope_json, '$.asset_type') AS asset_type,
    LOWER(json_extract_string(structured_scope_json, '$.max_severity')) AS max_severity
FROM bronze;
""")

for tabela in ["stg_reports", "stg_reporter", "stg_team", "stg_weakness", "stg_asset"]:
    total = con.sql(f"SELECT COUNT(*) FROM {tabela}").fetchone()[0]
    print(f"{tabela}: {total:,} linhas")
